## Batch scripts: Feature Importance (SHAP by default, permutation as validation option, + ablation, SomaScan -> Olink)

In [ ]:
# parameters
# batch_number = 0

# feature importance method: "shap" is the default for the full ~8000-protein sweep
# (single deterministic tree pass per protein, no repeated refit/permute);
# Methods currently describes.
importance_method = "shap" 
shap_check_additivity = False 

# permutation_importance settings (only used when importance_method == "permutation")
n_repeats = 5
subsample_n = None 
random_state = 613
perm_n_jobs = 1 

# ablation grid + model settings
ks = [1, 2, 3, 4, 5, 7, 10, 15, 20, 50]
ablation_max_iter = 30
ablation_early_stopping = True
ablation_validation_fraction = 0.1
ablation_n_iter_no_change = 3

output_dir = "Results/feature_imp/soma_olink"
protlist_chunk_dir = "Results/feature_imp/protlists/"
protlist_prefix = "so_protlist_remaining"


In [ ]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.path.dirname("..."), "..")))
project_root = os.path.abspath(os.path.expanduser("..."))
sys.path.append(project_root)
os.chdir(project_root)

import warnings
import pickle
import joblib
import pandas as pd
import numpy as np
from sklearn.inspection import permutation_importance
import shap
from sklearn.ensemble import HistGradientBoostingRegressor
from scipy.stats import pearsonr

def fxn():
    warnings.warn("deprecated", DeprecationWarning)

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    fxn()

## Data loading (once per batch, not once per protein)

In [ ]:
### MESA data
MESA_soma = pd.read_csv("Data/final_datasets/MESA_SOMA_5k_log2scaled_nodup_final.tsv.gz", 
                        sep = "\t", compression = "gzip").drop(columns={"bmi"})

MESA_olink = pd.read_csv("Data/final_datasets/MESA_Olink_3k_log2_nodup_rm10na_knn_final.tsv.gz", 
                         sep = "\t", compression = "gzip")

MESA_soma = MESA_soma.set_index("id")
MESA_olink = MESA_olink.set_index("id")

In [ ]:
## SAME pickled splits used elsewhere in the pipeline, so results are directly comparable
train_index = joblib.load("Data/helper_data/train_index.pkl")
test_index = joblib.load("Data/helper_data/test_index.pkl")

MESA_soma_test = MESA_soma.loc[test_index]
MESA_olink_test = MESA_olink.loc[test_index]
MESA_soma_train = MESA_soma.loc[train_index]
MESA_olink_train = MESA_olink.loc[train_index]

### columns to determine if it is overlapping protein
soma_prots = MESA_soma.columns

In [ ]:
with open(f"{protlist_chunk_dir}/soma_olink_remaining.tsv") as f:
    protlist = [line.strip() for line in f if line.strip()]
print(len(protlist))

## Feature importance + ablation loop (SomaScan -> Olink)

In [ ]:
def make_ablation_model():
    # capped max_iter + early stopping: we only need a relative ranking across k,
    # not the same convergence tolerance as the main saved models
    return HistGradientBoostingRegressor(
        random_state=0,
        max_iter=ablation_max_iter,
        early_stopping=ablation_early_stopping,
        validation_fraction=ablation_validation_fraction,
        n_iter_no_change=ablation_n_iter_no_change,
    )


def compute_shap_importance(protein_model, X_perm):
    # Exact Tree SHAP, tree_path_dependent perturbation: no background dataset needed,
    # cost is ~one tree traversal per sample rather than n_features * n_repeats refits,
    # which is what makes this tractable across ~8000 protein models.
    explainer = shap.TreeExplainer(protein_model, feature_perturbation="tree_path_dependent")
    shap_values = explainer.shap_values(X_perm, check_additivity=shap_check_additivity)
    importance_mean = np.abs(shap_values).mean(axis=0)
    importance_std = np.abs(shap_values).std(axis=0)
    return importance_mean, importance_std


def compute_feature_importance_and_ablation(protein_name):
    model_path = f"Data/soma_trained_models/soma_olink/5k_log2scaled_soma_to_3kolink/{protein_name}_hgb.pkl"
    if not os.path.exists(model_path):
        print(f"skip protein {protein_name}: no trained model at {model_path}")
        return

    protein_model = joblib.load(model_path)
    y_true_full = MESA_olink_test[protein_name]

    # Subsample the test set used for importance (shared by both methods below): ranking is
    # stable well below the full N. This matters most for importance_method='permutation',
    # where cost scales linearly with n_samples * n_repeats * n_features;
    if subsample_n is not None and subsample_n < len(MESA_soma_test):
        rng = np.random.RandomState(random_state)
        sub_idx = rng.choice(MESA_soma_test.index, size=subsample_n, replace=False)
        X_perm = MESA_soma_test.loc[sub_idx]
        y_perm = y_true_full.loc[sub_idx]
    else:
        X_perm = MESA_soma_test
        y_perm = y_true_full

    if importance_method == "shap":
        importance_mean, importance_std = compute_shap_importance(protein_model, X_perm)
    elif importance_method == "permutation":
        result = permutation_importance(
            protein_model, X_perm, y_perm,
            scoring="r2",
            n_repeats=n_repeats,
            random_state=random_state,
            n_jobs=perm_n_jobs
        )
        importance_mean, importance_std = result.importances_mean, result.importances_std
    else:
        raise ValueError(f"unknown importance_method: {importance_method}")

    importance_df = pd.DataFrame({
        'feature': X_perm.columns,
        'importance_mean': importance_mean,
        'importance_std': importance_std
    }).sort_values(by='importance_mean', ascending=False)
    importance_df.to_csv(f"{output_dir}/{protein_name}_feature_imp_{importance_method}.tsv", index=False, sep="\t")

    rank = np.argsort(importance_mean)[::-1]
    model_scores = []
    corr_scores = []

    if protein_name in soma_prots:
        for k in ks:
            feats = rank[:k]
            model_k = make_ablation_model()
            model_k.fit(MESA_soma_train.iloc[:, feats], MESA_olink_train[protein_name])
            model_scores.append(
                pearsonr(model_k.predict(MESA_soma_test.iloc[:, feats]), MESA_olink_test[protein_name])[0]
            )
            corr_scores.append(
                pearsonr(model_k.predict(MESA_soma_test.iloc[:, feats]), MESA_soma_test[protein_name])[0]
            )
        score_df = pd.DataFrame({"k": ks, "model_score": model_scores, "corr_score": corr_scores})
    else:
        for k in ks:
            feats = rank[:k]
            model_k = make_ablation_model()
            model_k.fit(MESA_soma_train.iloc[:, feats], MESA_olink_train[protein_name])
            model_scores.append(
                pearsonr(model_k.predict(MESA_soma_test.iloc[:, feats]), MESA_olink_test[protein_name])[0]
            )
        score_df = pd.DataFrame({"k": ks, "model_score": model_scores, "corr_score": np.nan})

    score_df.to_csv(f"{output_dir}/{protein_name}_ablation_r2.tsv", index=False, sep="\t")
    print(f"done: {protein_name}")

In [ ]:
### small test before looping
# protlist = protlist[:5]

In [ ]:
%%time

for p in protlist:
    print(f"processing: {p}")
    try:
        compute_feature_importance_and_ablation(p)
    except Exception as e:
        print(f"ERROR on protein {p}: {e}")
        continue